# Objective 3 — Step 8: Frozen Hybrid Ensemble Replication

Step 7 selected the final German-developed ensemble architecture **using German evidence only**.

## Frozen hybrid architecture

**Feature selection**
- Group-aware Chi-Square
- Top 75% of original/source features
- Re-fitted only on each outer-training partition

**Balanced heterogeneous base learners**
1. Logistic Regression — `class_weight="balanced"`
2. Random Forest — `class_weight="balanced"`
3. XGBoost — `scale_pos_weight = N_negative / N_positive` from the current outer-training partition

**Fusion**
- Equal-probability Soft Voting
- Final probability = mean of LR, RF, and XGBoost positive-class probabilities

**Decision threshold**
- 0.50 in this replication stage

## Replication datasets

- Australian Credit Approval
- Taiwan Credit Card Default

The architecture, feature-retention proportion, base learners, weighting rule, fusion rule, XGBoost structure, and threshold are **not retuned** on either replication dataset.

This notebook also verifies that the XGBoost component exactly reproduces the Step-6 FrozenChi2Top75 + Balanced XGBoost result on the same folds.


In [1]:
%pip install pandas numpy scikit-learn xgboost

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.2
[notice] To update, run: C:\Users\hp\AppData\Local\Programs\Python\Python311\python.exe -m pip install --upgrade pip


In [2]:

from pathlib import Path
from itertools import combinations
import json
import math
import time
import warnings

import numpy as np
import pandas as pd

from IPython.display import display

from sklearn.compose import ColumnTransformer
from sklearn.feature_selection import chi2
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    matthews_corrcoef,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

warnings.filterwarnings("ignore")

BASE_DIR = Path(r"D:\PHD\Research Paper writing\3rd Obj. paper")

STEP2_DIR = BASE_DIR / "results" / "preprocessing_protocol"
BASELINE_DIR = BASE_DIR / "results" / "baseline_models"
STEP6_DIR = BASE_DIR / "results" / "cost_sensitive_xgb_ablation"
STEP7_DIR = BASE_DIR / "results" / "ensemble_german_development"
DATA_DIR = BASE_DIR / "data" / "processed"

OUT_DIR = BASE_DIR / "results" / "frozen_hybrid_ensemble_replication"
OUT_DIR.mkdir(parents=True, exist_ok=True)

DICTIONARY_FILE = STEP2_DIR / "preprocessing_data_dictionary.csv"
BASELINE_RESULTS_FILE = BASELINE_DIR / "baseline_fold_results_all.csv"
BASELINE_PREDICTIONS_FILE = BASELINE_DIR / "baseline_predictions_all.csv"
STEP6_RESULTS_FILE = STEP6_DIR / "cost_sensitive_xgb_fold_results.csv"
STEP7_DECISION_FILE = STEP7_DIR / "german_ensemble_decision_table.csv"

DATASET_FILES = {
    "Australian Credit Approval":
        DATA_DIR / "australian_credit_approval_cleaned.csv",
    "Taiwan Credit Card Default":
        DATA_DIR / "taiwan_credit_card_default_cleaned.csv",
}

required = [
    DICTIONARY_FILE,
    BASELINE_RESULTS_FILE,
    BASELINE_PREDICTIONS_FILE,
    STEP6_RESULTS_FILE,
    STEP7_DECISION_FILE,
    *DATASET_FILES.values(),
]

missing = [str(p) for p in required if not p.exists()]
if missing:
    raise FileNotFoundError(
        "Previous-step files are missing:\n" + "\n".join(missing)
    )

REPEAT_SEEDS = [42, 142, 242, 342, 442]
N_FOLDS = 5
FROZEN_FRACTION = 0.75
BASE_MODELS = ["LR", "RF", "XGB"]

print("Output folder:", OUT_DIR)


Output folder: D:\PHD\Research Paper writing\3rd Obj. paper\results\frozen_hybrid_ensemble_replication


## 1. Lock the Step-7 German-developed architecture

In [3]:

step7_decision = pd.read_csv(STEP7_DECISION_FILE)

selected_german = step7_decision[
    (step7_decision["feature_regime"] == "FrozenChi2Top75")
    & (step7_decision["training_regime"] == "Balanced")
    & (step7_decision["ensemble_type"] == "SoftVote")
].copy()

if len(selected_german) != 1:
    raise RuntimeError(
        "The frozen Step-7 architecture could not be uniquely identified."
    )

frozen_architecture = {
    "development_dataset": "German Credit",
    "feature_selection": "Group-aware Chi-Square Top75",
    "retained_source_feature_fraction": 0.75,
    "training_regime": "Balanced",
    "base_learners": BASE_MODELS,
    "fusion_rule": "Equal-probability soft voting",
    "classification_threshold": 0.50,
    "external_replication_datasets": [
        "Australian Credit Approval",
        "Taiwan Credit Card Default",
    ],
    "external_retuning_permitted": False,
    "german_development_metrics": {
        "roc_auc": float(selected_german["ROC_AUC"].iloc[0]),
        "pr_auc": float(selected_german["PR_AUC"].iloc[0]),
        "recall": float(selected_german["Recall"].iloc[0]),
        "precision": float(selected_german["Precision"].iloc[0]),
        "f1": float(selected_german["F1"].iloc[0]),
        "balanced_accuracy": float(
            selected_german["Balanced_Accuracy"].iloc[0]
        ),
        "mcc": float(selected_german["MCC"].iloc[0]),
    },
}

with open(
    OUT_DIR / "frozen_hybrid_architecture.json",
    "w",
    encoding="utf-8",
) as f:
    json.dump(frozen_architecture, f, indent=4)

print(json.dumps(frozen_architecture, indent=2))


{
  "development_dataset": "German Credit",
  "feature_selection": "Group-aware Chi-Square Top75",
  "retained_source_feature_fraction": 0.75,
  "training_regime": "Balanced",
  "base_learners": [
    "LR",
    "RF",
    "XGB"
  ],
  "fusion_rule": "Equal-probability soft voting",
  "classification_threshold": 0.5,
  "external_replication_datasets": [
    "Australian Credit Approval",
    "Taiwan Credit Card Default"
  ],
  "external_retuning_permitted": false,
  "german_development_metrics": {
    "roc_auc": 0.7988015059442607,
    "pr_auc": 0.6348971004203964,
    "recall": 0.6265063083613087,
    "precision": 0.5865047616054124,
    "f1": 0.6030748803699565,
    "balanced_accuracy": 0.7183547481403739,
    "mcc": 0.4285895190336244
  }
}


## 2. Load replication datasets and feature roles

In [4]:

dictionary = pd.read_csv(DICTIONARY_FILE)
baseline_results = pd.read_csv(BASELINE_RESULTS_FILE)
baseline_predictions = pd.read_csv(BASELINE_PREDICTIONS_FILE)
step6_results = pd.read_csv(STEP6_RESULTS_FILE)

datasets = {
    name: pd.read_csv(path)
    for name, path in DATASET_FILES.items()
}

def roles_for(dataset_name):
    d = dictionary[dictionary["dataset"] == dataset_name].copy()

    return {
        role: (
            d.loc[d["role"] == role, "variable"]
            .astype(str)
            .tolist()
        )
        for role in [
            "categorical",
            "ordinal",
            "numerical",
            "identifier",
        ]
    }

roles = {
    name: roles_for(name)
    for name in datasets
}

for dataset_name, df in datasets.items():
    r = roles[dataset_name]
    predictors = (
        r["categorical"]
        + r["ordinal"]
        + r["numerical"]
    )

    print(
        dataset_name,
        "| N =", len(df),
        "| predictors =", len(predictors),
        "| frozen Top75 =", math.ceil(
            len(predictors) * FROZEN_FRACTION
        ),
        "| adverse rate =", round(
            df["adverse_target"].mean(), 4
        ),
    )


Australian Credit Approval | N = 690 | predictors = 14 | frozen Top75 = 11 | adverse rate = 0.5551
Taiwan Credit Card Default | N = 30000 | predictors = 23 | frozen Top75 = 18 | adverse rate = 0.2212


## 3. Fold-wise preprocessing

In [5]:

def make_one_hot_encoder():
    try:
        return OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=False,
            dtype=np.float32,
        )
    except TypeError:
        return OneHotEncoder(
            handle_unknown="ignore",
            sparse=False,
            dtype=np.float32,
        )


def build_preprocessor(
    dataset_name,
    mode,
    selected_features,
):
    r = roles[dataset_name]
    selected_features = list(selected_features)

    selected_cat = [
        f for f in r["categorical"]
        if f in selected_features
    ]
    selected_ord = [
        f for f in r["ordinal"]
        if f in selected_features
    ]
    selected_num = [
        f for f in r["numerical"]
        if f in selected_features
    ]

    transformers = []

    if selected_num:
        if mode == "scaled":
            num_pipe = Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler()),
            ])
        elif mode == "tree":
            num_pipe = Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
            ])
        elif mode == "chi2":
            num_pipe = Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", MinMaxScaler(clip=True)),
            ])
        else:
            raise ValueError(mode)

        transformers.append(
            ("num", num_pipe, selected_num)
        )

    if selected_ord:
        if mode == "scaled":
            ord_pipe = Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("scaler", StandardScaler()),
            ])
        elif mode == "tree":
            ord_pipe = Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
            ])
        elif mode == "chi2":
            ord_pipe = Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("scaler", MinMaxScaler(clip=True)),
            ])
        else:
            raise ValueError(mode)

        transformers.append(
            ("ord", ord_pipe, selected_ord)
        )

    if selected_cat:
        cat_pipe = Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", make_one_hot_encoder()),
        ])

        transformers.append(
            ("cat", cat_pipe, selected_cat)
        )

    return ColumnTransformer(
        transformers=transformers,
        remainder="drop",
        verbose_feature_names_out=True,
    )


## 4. Group-aware frozen Chi-Square Top-75%

In [6]:

def feature_map_from_fitted_chi2_preprocessor(
    prep,
    dataset_name,
):
    r = roles[dataset_name]
    rows = []
    idx = 0

    for feature in r["numerical"]:
        rows.append({
            "transformed_index": idx,
            "source_feature": feature,
            "source_role": "numerical",
        })
        idx += 1

    for feature in r["ordinal"]:
        rows.append({
            "transformed_index": idx,
            "source_feature": feature,
            "source_role": "ordinal",
        })
        idx += 1

    if r["categorical"]:
        cat_pipe = prep.named_transformers_["cat"]
        encoder = cat_pipe.named_steps["onehot"]

        for source_feature, categories in zip(
            r["categorical"],
            encoder.categories_,
        ):
            for _ in categories:
                rows.append({
                    "transformed_index": idx,
                    "source_feature": source_feature,
                    "source_role": "categorical",
                })
                idx += 1

    fmap = pd.DataFrame(rows)

    assert len(fmap) == len(
        prep.get_feature_names_out()
    )

    return fmap


def group_aware_chi2_top75(
    dataset_name,
    X_train,
    y_train,
):
    r = roles[dataset_name]
    all_features = (
        r["categorical"]
        + r["ordinal"]
        + r["numerical"]
    )

    prep = build_preprocessor(
        dataset_name,
        "chi2",
        all_features,
    )

    X_chi = prep.fit_transform(
        X_train[all_features],
        y_train,
    )

    assert np.asarray(X_chi).min() >= -1e-12

    fmap = feature_map_from_fitted_chi2_preprocessor(
        prep,
        dataset_name,
    )

    raw_scores, _ = chi2(
        X_chi,
        y_train,
    )

    temp = fmap.copy()
    temp["raw_score"] = (
        pd.Series(raw_scores)
        .replace([np.inf, -np.inf], np.nan)
        .fillna(0.0)
        .to_numpy()
    )

    n = len(temp)

    ranks = temp["raw_score"].rank(
        ascending=False,
        method="average",
    )

    if n > 1:
        temp["normalized_relevance"] = (
            1.0
            - (ranks - 1.0) / (n - 1.0)
        )
    else:
        temp["normalized_relevance"] = 1.0

    grouped = (
        temp.groupby(
            "source_feature",
            as_index=False,
        )
        .agg(
            group_score=(
                "normalized_relevance",
                "mean",
            )
        )
    )

    grouped["source_rank"] = (
        grouped["group_score"]
        .rank(
            ascending=False,
            method="average",
        )
    )

    grouped = grouped.sort_values(
        [
            "source_rank",
            "source_feature",
        ]
    ).reset_index(drop=True)

    n_select = int(
        math.ceil(
            len(grouped)
            * FROZEN_FRACTION
        )
    )

    selected = (
        grouped["source_feature"]
        .astype(str)
        .tolist()[:n_select]
    )

    return selected, grouped


## 5. Frozen balanced base learners

In [7]:

def balanced_ratio(y_train):
    n_positive = int(
        (y_train == 1).sum()
    )
    n_negative = int(
        (y_train == 0).sum()
    )

    return n_negative / n_positive


def build_balanced_base_pipeline(
    dataset_name,
    model_name,
    selected_features,
    y_train,
):
    if model_name == "LR":
        estimator = LogisticRegression(
            max_iter=3000,
            solver="lbfgs",
            class_weight="balanced",
            random_state=42,
        )
        mode = "scaled"

    elif model_name == "RF":
        estimator = RandomForestClassifier(
            n_estimators=300,
            class_weight="balanced",
            random_state=42,
            n_jobs=-1,
        )
        mode = "tree"

    elif model_name == "XGB":
        estimator = XGBClassifier(
            n_estimators=300,
            max_depth=4,
            learning_rate=0.05,
            subsample=0.9,
            colsample_bytree=0.9,
            objective="binary:logistic",
            eval_metric="logloss",
            scale_pos_weight=float(
                balanced_ratio(y_train)
            ),
            random_state=42,
            n_jobs=-1,
            verbosity=0,
        )
        mode = "tree"

    else:
        raise ValueError(model_name)

    return Pipeline([
        (
            "preprocessor",
            build_preprocessor(
                dataset_name,
                mode,
                selected_features,
            ),
        ),
        ("model", estimator),
    ])


## 6. Metrics

In [8]:

def calculate_metrics(
    y_true,
    y_pred,
    y_score,
):
    tn, fp, fn, tp = confusion_matrix(
        y_true,
        y_pred,
        labels=[0, 1],
    ).ravel()

    specificity = (
        tn / (tn + fp)
        if (tn + fp) > 0
        else np.nan
    )

    sensitivity = (
        tp / (tp + fn)
        if (tp + fn) > 0
        else np.nan
    )

    gmean = (
        math.sqrt(
            specificity
            * sensitivity
        )
        if not np.isnan(
            specificity + sensitivity
        )
        else np.nan
    )

    fpr, tpr, _ = roc_curve(
        y_true,
        y_score,
    )

    ks = float(
        np.max(tpr - fpr)
    )

    n = len(y_true)

    cost_metrics = {}

    for fn_cost in [1, 2, 5, 10]:
        total_cost = (
            fp
            + fn_cost * fn
        )

        cost_metrics[
            f"cost_FN{fn_cost}_FP1_per100"
        ] = (
            100.0
            * total_cost
            / n
        )

    return {
        "accuracy": accuracy_score(
            y_true,
            y_pred,
        ),
        "precision_adverse": precision_score(
            y_true,
            y_pred,
            pos_label=1,
            zero_division=0,
        ),
        "recall_adverse": recall_score(
            y_true,
            y_pred,
            pos_label=1,
            zero_division=0,
        ),
        "specificity": specificity,
        "f1_adverse": f1_score(
            y_true,
            y_pred,
            pos_label=1,
            zero_division=0,
        ),
        "balanced_accuracy": balanced_accuracy_score(
            y_true,
            y_pred,
        ),
        "mcc": matthews_corrcoef(
            y_true,
            y_pred,
        ),
        "roc_auc": roc_auc_score(
            y_true,
            y_score,
        ),
        "pr_auc": average_precision_score(
            y_true,
            y_score,
        ),
        "gmean": gmean,
        "ks_statistic": ks,
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
        **cost_metrics,
    }


## 7. Recreate and verify the exact Step-3 outer folds

In [9]:

def build_verified_splits(
    dataset_name,
    df,
):
    r = roles[dataset_name]

    features = (
        r["categorical"]
        + r["ordinal"]
        + r["numerical"]
    )

    X_local = df[features].copy()
    y_local = (
        df["adverse_target"]
        .astype(int)
        .copy()
    )
    groups_local = (
        df["profile_group_id"]
        .astype(str)
        .copy()
    )

    saved = baseline_predictions[
        (
            baseline_predictions["dataset"]
            == dataset_name
        )
        & (
            baseline_predictions["model"]
            == "XGB"
        )
    ].copy()

    split_dict = {}

    for repeat_no, seed in enumerate(
        REPEAT_SEEDS,
        start=1,
    ):
        splitter = StratifiedGroupKFold(
            n_splits=N_FOLDS,
            shuffle=True,
            random_state=seed,
        )

        for fold_no, (
            train_idx,
            test_idx,
        ) in enumerate(
            splitter.split(
                X_local,
                y_local,
                groups_local,
            ),
            start=1,
        ):
            run_id = (
                f"R{repeat_no}_F{fold_no}"
            )

            expected_test = set(
                np.asarray(
                    test_idx,
                    dtype=int,
                ).tolist()
            )

            saved_test = set(
                saved.loc[
                    saved["run_id"]
                    == run_id,
                    "source_row_index",
                ]
                .astype(int)
                .tolist()
            )

            assert expected_test == saved_test

            train_groups = set(
                groups_local.iloc[
                    train_idx
                ]
            )
            test_groups = set(
                groups_local.iloc[
                    test_idx
                ]
            )

            assert len(
                train_groups
                .intersection(
                    test_groups
                )
            ) == 0

            split_dict[run_id] = {
                "repeat": repeat_no,
                "fold": fold_no,
                "seed": seed,
                "train_idx": np.asarray(
                    train_idx,
                    dtype=int,
                ),
                "test_idx": np.asarray(
                    test_idx,
                    dtype=int,
                ),
            }

    return split_dict


all_splits = {
    dataset_name: build_verified_splits(
        dataset_name,
        df,
    )
    for dataset_name, df
    in datasets.items()
}

print(
    "All Australian and Taiwan folds "
    "exactly match Step 3."
)


All Australian and Taiwan folds exactly match Step 3.


## 8. Run the frozen hybrid ensemble

In [10]:

result_rows = []
prediction_rows = []
selection_rows = []
base_metric_rows = []

for dataset_name, df in datasets.items():
    r = roles[dataset_name]

    all_features_local = (
        r["categorical"]
        + r["ordinal"]
        + r["numerical"]
    )

    X_local = df[
        all_features_local
    ].copy()

    y_local = (
        df["adverse_target"]
        .astype(int)
        .copy()
    )

    print("\n" + "=" * 80)
    print(dataset_name)
    print("=" * 80)

    for run_number, (
        run_id,
        info,
    ) in enumerate(
        all_splits[
            dataset_name
        ].items(),
        start=1,
    ):
        train_idx = info["train_idx"]
        test_idx = info["test_idx"]

        X_train = X_local.iloc[
            train_idx
        ].copy()

        X_test = X_local.iloc[
            test_idx
        ].copy()

        y_train = y_local.iloc[
            train_idx
        ].copy()

        y_test = y_local.iloc[
            test_idx
        ].copy()

        fs_start = time.perf_counter()

        selected_features, ranking = (
            group_aware_chi2_top75(
                dataset_name,
                X_train,
                y_train,
            )
        )

        fs_runtime = (
            time.perf_counter()
            - fs_start
        )

        selection_rows.append({
            "dataset": dataset_name,
            "run_id": run_id,
            "repeat": info["repeat"],
            "fold": info["fold"],
            "total_source_features": len(
                all_features_local
            ),
            "selected_source_features": len(
                selected_features
            ),
            "feature_reduction_pct": (
                100.0
                * (
                    1.0
                    - len(
                        selected_features
                    )
                    / len(
                        all_features_local
                    )
                )
            ),
            "selected_features": ";".join(
                selected_features
            ),
            "feature_selection_runtime_seconds": (
                fs_runtime
            ),
        })

        base_probabilities = {}
        base_predictions = {}
        base_runtimes = {}

        for model_name in BASE_MODELS:
            pipe = build_balanced_base_pipeline(
                dataset_name=dataset_name,
                model_name=model_name,
                selected_features=selected_features,
                y_train=y_train,
            )

            start = time.perf_counter()

            pipe.fit(
                X_train[
                    selected_features
                ],
                y_train,
            )

            probabilities = (
                pipe.predict_proba(
                    X_test[
                        selected_features
                    ]
                )[:, 1]
            )

            runtime = (
                time.perf_counter()
                - start
            )

            predictions_binary = (
                probabilities >= 0.50
            ).astype(int)

            base_probabilities[
                model_name
            ] = probabilities

            base_predictions[
                model_name
            ] = predictions_binary

            base_runtimes[
                model_name
            ] = runtime

            base_metrics = (
                calculate_metrics(
                    y_test,
                    predictions_binary,
                    probabilities,
                )
            )

            base_metric_rows.append({
                "dataset": dataset_name,
                "run_id": run_id,
                "repeat": info["repeat"],
                "fold": info["fold"],
                "model": model_name,
                "selected_source_features": len(
                    selected_features
                ),
                "runtime_seconds": runtime,
                **base_metrics,
            })

        probability_matrix = np.column_stack(
            [
                base_probabilities[
                    model_name
                ]
                for model_name
                in BASE_MODELS
            ]
        )

        hybrid_score = (
            probability_matrix.mean(
                axis=1
            )
        )

        hybrid_pred = (
            hybrid_score >= 0.50
        ).astype(int)

        hybrid_metrics = (
            calculate_metrics(
                y_test,
                hybrid_pred,
                hybrid_score,
            )
        )

        result_rows.append({
            "dataset": dataset_name,
            "run_id": run_id,
            "repeat": info["repeat"],
            "fold": info["fold"],
            "feature_regime": (
                "FrozenChi2Top75"
            ),
            "training_regime": (
                "Balanced"
            ),
            "ensemble_type": (
                "SoftVote"
            ),
            "selected_source_features": len(
                selected_features
            ),
            "feature_selection_runtime_seconds": (
                fs_runtime
            ),
            "lr_runtime_seconds": (
                base_runtimes["LR"]
            ),
            "rf_runtime_seconds": (
                base_runtimes["RF"]
            ),
            "xgb_runtime_seconds": (
                base_runtimes["XGB"]
            ),
            **hybrid_metrics,
        })

        for local_position, source_index in enumerate(
            test_idx
        ):
            prediction_rows.append({
                "dataset": dataset_name,
                "run_id": run_id,
                "repeat": info["repeat"],
                "fold": info["fold"],
                "source_row_index": int(
                    source_index
                ),
                "y_true": int(
                    y_test.iloc[
                        local_position
                    ]
                ),
                "lr_probability": float(
                    base_probabilities[
                        "LR"
                    ][
                        local_position
                    ]
                ),
                "rf_probability": float(
                    base_probabilities[
                        "RF"
                    ][
                        local_position
                    ]
                ),
                "xgb_probability": float(
                    base_probabilities[
                        "XGB"
                    ][
                        local_position
                    ]
                ),
                "hybrid_probability": float(
                    hybrid_score[
                        local_position
                    ]
                ),
                "hybrid_prediction_0_5": int(
                    hybrid_pred[
                        local_position
                    ]
                ),
            })

        print(
            f"{run_id} ({run_number}/25) | "
            f"ROC={hybrid_metrics['roc_auc']:.4f} | "
            f"PR={hybrid_metrics['pr_auc']:.4f} | "
            f"Recall={hybrid_metrics['recall_adverse']:.4f} | "
            f"F1={hybrid_metrics['f1_adverse']:.4f} | "
            f"MCC={hybrid_metrics['mcc']:.4f}"
        )


results = pd.DataFrame(
    result_rows
)

predictions = pd.DataFrame(
    prediction_rows
)

selected_sets = pd.DataFrame(
    selection_rows
)

base_metrics = pd.DataFrame(
    base_metric_rows
)

results.to_csv(
    OUT_DIR
    / "frozen_hybrid_fold_results.csv",
    index=False,
)

predictions.to_csv(
    OUT_DIR
    / "frozen_hybrid_outer_predictions.csv",
    index=False,
)

selected_sets.to_csv(
    OUT_DIR
    / "frozen_hybrid_selected_feature_sets.csv",
    index=False,
)

base_metrics.to_csv(
    OUT_DIR
    / "frozen_hybrid_base_model_fold_results.csv",
    index=False,
)

print(
    "\nFrozen hybrid replication completed."
)



Australian Credit Approval
R1_F1 (1/25) | ROC=0.9194 | PR=0.9492 | Recall=0.7901 | F1=0.8477 | MCC=0.6745
R1_F2 (2/25) | ROC=0.9361 | PR=0.9478 | Recall=0.8765 | F1=0.8987 | MCC=0.7647
R1_F3 (3/25) | ROC=0.9341 | PR=0.9342 | Recall=0.8875 | F1=0.8987 | MCC=0.7635
R1_F4 (4/25) | ROC=0.9220 | PR=0.8932 | Recall=0.8955 | F1=0.8571 | MCC=0.7133
R1_F5 (5/25) | ROC=0.9521 | PR=0.9585 | Recall=0.8514 | F1=0.9000 | MCC=0.8031
R2_F1 (6/25) | ROC=0.9106 | PR=0.9217 | Recall=0.8356 | F1=0.8472 | MCC=0.6809
R2_F2 (7/25) | ROC=0.8993 | PR=0.9248 | Recall=0.8161 | F1=0.8712 | MCC=0.6968
R2_F3 (8/25) | ROC=0.9280 | PR=0.9293 | Recall=0.8472 | F1=0.8714 | MCC=0.7405
R2_F4 (9/25) | ROC=0.9244 | PR=0.9163 | Recall=0.8667 | F1=0.8784 | MCC=0.7381
R2_F5 (10/25) | ROC=0.9817 | PR=0.9864 | Recall=0.9211 | F1=0.9211 | MCC=0.8243
R3_F1 (11/25) | ROC=0.9127 | PR=0.9270 | Recall=0.8056 | F1=0.8286 | MCC=0.6535
R3_F2 (12/25) | ROC=0.9181 | PR=0.9150 | Recall=0.8421 | F1=0.8707 | MCC=0.7257
R3_F3 (13/25) | ROC=0

## 9. Verify the XGBoost component against Step 6

In [11]:

reference_xgb = step6_results[
    (
        step6_results["dataset"]
        .isin(
            list(
                datasets.keys()
            )
        )
    )
    & (
        step6_results[
            "feature_regime"
        ]
        == "FrozenChi2Top75"
    )
    & (
        step6_results[
            "weight_strategy"
        ]
        == "Balanced"
    )
][
    [
        "dataset",
        "run_id",
        "roc_auc",
        "pr_auc",
        "recall_adverse",
        "f1_adverse",
        "balanced_accuracy",
        "mcc",
    ]
].copy()

current_xgb = base_metrics[
    base_metrics["model"]
    == "XGB"
][
    [
        "dataset",
        "run_id",
        "roc_auc",
        "pr_auc",
        "recall_adverse",
        "f1_adverse",
        "balanced_accuracy",
        "mcc",
    ]
].copy()

check = current_xgb.merge(
    reference_xgb,
    on=[
        "dataset",
        "run_id",
    ],
    suffixes=(
        "_step8",
        "_step6",
    ),
    validate="one_to_one",
)

for metric in [
    "roc_auc",
    "pr_auc",
    "recall_adverse",
    "f1_adverse",
    "balanced_accuracy",
    "mcc",
]:
    maximum_difference = (
        np.abs(
            check[
                f"{metric}_step8"
            ]
            - check[
                f"{metric}_step6"
            ]
        ).max()
    )

    print(
        metric,
        "max absolute difference =",
        maximum_difference,
    )

    assert (
        maximum_difference
        < 1e-10
    )

print(
    "\nXGBoost component exactly "
    "reproduces Step 6."
)


roc_auc max absolute difference = 1.1102230246251565e-16
pr_auc max absolute difference = 1.1102230246251565e-16
recall_adverse max absolute difference = 1.1102230246251565e-16
f1_adverse max absolute difference = 0.0
balanced_accuracy max absolute difference = 1.1102230246251565e-16
mcc max absolute difference = 5.551115123125783e-17

XGBoost component exactly reproduces Step 6.


## 10. External replication summary

In [12]:

summary = (
    results.groupby(
        "dataset",
        as_index=False,
    )
    .agg(
        Selected_Source_Features=(
            "selected_source_features",
            "mean",
        ),
        ROC_AUC=("roc_auc", "mean"),
        ROC_AUC_SD=("roc_auc", "std"),
        PR_AUC=("pr_auc", "mean"),
        PR_AUC_SD=("pr_auc", "std"),
        Recall=("recall_adverse", "mean"),
        Precision=("precision_adverse", "mean"),
        F1=("f1_adverse", "mean"),
        Balanced_Accuracy=(
            "balanced_accuracy",
            "mean",
        ),
        MCC=("mcc", "mean"),
        MCC_SD=("mcc", "std"),
        GMean=("gmean", "mean"),
        KS=("ks_statistic", "mean"),
        Cost_2_1=(
            "cost_FN2_FP1_per100",
            "mean",
        ),
        Cost_5_1=(
            "cost_FN5_FP1_per100",
            "mean",
        ),
        Cost_10_1=(
            "cost_FN10_FP1_per100",
            "mean",
        ),
    )
)

summary.to_csv(
    OUT_DIR
    / "frozen_hybrid_replication_summary.csv",
    index=False,
)

display(summary)


,dataset,Selected_Source_Features,ROC_AUC,ROC_AUC_SD,PR_AUC,PR_AUC_SD,Recall,Precision,F1,Balanced_Accuracy,MCC,MCC_SD,GMean,KS,Cost_2_1,Cost_5_1,Cost_10_1
0,Australian Credit Approval,11.0,0.931624,0.017534,0.938131,0.022227,0.856447,0.899657,0.876949,0.869179,0.734673,0.051039,0.868768,0.775629,21.217391,45.130435,84.985507
1,Taiwan Credit Card Default,18.0,0.777340,0.006228,0.557204,0.013392,0.509745,0.573779,0.539777,0.701085,0.420070,0.012298,0.674412,0.427319,30.068667,62.604709,116.831446


## 11. Compare the frozen hybrid with key XGBoost references

In [13]:

reference_configs = []

# Step-3 all-feature unweighted XGBoost
ref_step3 = baseline_results[
    (
        baseline_results["dataset"]
        .isin(
            list(
                datasets.keys()
            )
        )
    )
    & (
        baseline_results["model"]
        == "XGB"
    )
].copy()

ref_step3["reference_name"] = (
    "AllFeatures_Unweighted_XGB"
)

reference_configs.append(
    ref_step3
)

# Step-6 all-feature balanced XGBoost
ref_all_bal = step6_results[
    (
        step6_results["dataset"]
        .isin(
            list(
                datasets.keys()
            )
        )
    )
    & (
        step6_results["feature_regime"]
        == "AllFeatures"
    )
    & (
        step6_results["weight_strategy"]
        == "Balanced"
    )
].copy()

ref_all_bal["reference_name"] = (
    "AllFeatures_Balanced_XGB"
)

reference_configs.append(
    ref_all_bal
)

# Step-6 frozen-feature balanced XGBoost
ref_fs_bal = step6_results[
    (
        step6_results["dataset"]
        .isin(
            list(
                datasets.keys()
            )
        )
    )
    & (
        step6_results["feature_regime"]
        == "FrozenChi2Top75"
    )
    & (
        step6_results["weight_strategy"]
        == "Balanced"
    )
].copy()

ref_fs_bal["reference_name"] = (
    "FrozenFS_Balanced_XGB"
)

reference_configs.append(
    ref_fs_bal
)

reference_table = pd.concat(
    reference_configs,
    ignore_index=True,
    sort=False,
)

comparison_metrics = [
    "roc_auc",
    "pr_auc",
    "recall_adverse",
    "precision_adverse",
    "f1_adverse",
    "balanced_accuracy",
    "mcc",
    "gmean",
    "ks_statistic",
    "cost_FN2_FP1_per100",
    "cost_FN5_FP1_per100",
    "cost_FN10_FP1_per100",
]

paired_rows = []

for reference_name, reference_group in (
    reference_table.groupby(
        "reference_name"
    )
):
    ref_columns = [
        "dataset",
        "run_id",
        *comparison_metrics,
    ]

    ref = reference_group[
        ref_columns
    ].copy()

    ref = ref.rename(
        columns={
            metric: (
                "reference_"
                + metric
            )
            for metric
            in comparison_metrics
        }
    )

    merged = results.merge(
        ref,
        on=[
            "dataset",
            "run_id",
        ],
        how="left",
        validate="one_to_one",
    )

    for metric in comparison_metrics:
        merged[
            "delta_" + metric
        ] = (
            merged[metric]
            - merged[
                "reference_"
                + metric
            ]
        )

    merged[
        "reference_name"
    ] = reference_name

    paired_rows.append(
        merged
    )

paired = pd.concat(
    paired_rows,
    ignore_index=True,
)

paired.to_csv(
    OUT_DIR
    / "frozen_hybrid_paired_deltas.csv",
    index=False,
)

delta_summary = (
    paired.groupby(
        [
            "dataset",
            "reference_name",
        ],
        as_index=False,
    )
    .agg(
        Delta_ROC_AUC=(
            "delta_roc_auc",
            "mean",
        ),
        Delta_PR_AUC=(
            "delta_pr_auc",
            "mean",
        ),
        Delta_Recall=(
            "delta_recall_adverse",
            "mean",
        ),
        Delta_Precision=(
            "delta_precision_adverse",
            "mean",
        ),
        Delta_F1=(
            "delta_f1_adverse",
            "mean",
        ),
        Delta_Balanced_Accuracy=(
            "delta_balanced_accuracy",
            "mean",
        ),
        Delta_MCC=(
            "delta_mcc",
            "mean",
        ),
        Delta_Cost_2_1=(
            "delta_cost_FN2_FP1_per100",
            "mean",
        ),
        Delta_Cost_5_1=(
            "delta_cost_FN5_FP1_per100",
            "mean",
        ),
        Delta_Cost_10_1=(
            "delta_cost_FN10_FP1_per100",
            "mean",
        ),
    )
)

delta_summary.to_csv(
    OUT_DIR
    / "frozen_hybrid_delta_summary.csv",
    index=False,
)

display(delta_summary)


,dataset,reference_name,Delta_ROC_AUC,Delta_PR_AUC,Delta_Recall,Delta_Precision,Delta_F1,Delta_Balanced_Accuracy,Delta_MCC,Delta_Cost_2_1,Delta_Cost_5_1,Delta_Cost_10_1
0,Australian Credit Approval,AllFeatures_Balanced_XGB,-0.000633,0.005057,-0.021389,0.006239,-0.007897,-0.005211,-0.011747,1.797101,5.275362,11.072464
1,Australian Credit Approval,AllFeatures_Unweighted_XGB,0.000056,0.005295,-0.022510,0.010304,-0.006387,-0.002951,-0.007659,NaN,NaN,NaN
2,Australian Credit Approval,FrozenFS_Balanced_XGB,0.005459,0.008340,-0.004773,0.015373,0.004985,0.008897,0.016283,-0.463768,0.318841,1.623188
3,Taiwan Credit Card Default,AllFeatures_Balanced_XGB,-0.005639,-0.002891,-0.119154,0.102498,0.001071,-0.013125,0.030201,-1.963310,5.944607,19.124469
4,Taiwan Credit Card Default,AllFeatures_Unweighted_XGB,-0.005894,-0.003321,0.141781,-0.097768,0.064444,0.042669,0.017159,NaN,NaN,NaN
5,Taiwan Credit Card Default,FrozenFS_Balanced_XGB,-0.005211,-0.002037,-0.120123,0.103043,0.001094,-0.013220,0.030322,-1.981301,5.990593,19.277084


## 12. Feature-selection stability in the frozen hybrid

In [14]:

def parse_set(value):
    return set(
        str(value).split(";")
    )


stability_rows = []
frequency_rows = []

for dataset_name, group in selected_sets.groupby(
    "dataset"
):
    sets = [
        parse_set(value)
        for value in group[
            "selected_features"
        ]
    ]

    pair_scores = []

    for set_a, set_b in combinations(
        sets,
        2,
    ):
        union = set_a | set_b

        pair_scores.append(
            len(set_a & set_b)
            / len(union)
            if union
            else 1.0
        )

    stability_rows.append({
        "dataset": dataset_name,
        "mean_jaccard": np.mean(
            pair_scores
        ),
        "std_jaccard": np.std(
            pair_scores,
            ddof=1,
        ),
        "min_jaccard": np.min(
            pair_scores
        ),
        "max_jaccard": np.max(
            pair_scores
        ),
        "pairwise_comparisons": len(
            pair_scores
        ),
    })

    r = roles[dataset_name]
    all_features_local = (
        r["categorical"]
        + r["ordinal"]
        + r["numerical"]
    )

    for feature in all_features_local:
        count = sum(
            feature in selected_set
            for selected_set in sets
        )

        frequency_rows.append({
            "dataset": dataset_name,
            "source_feature": feature,
            "selected_runs": count,
            "total_runs": len(sets),
            "selection_frequency": (
                count
                / len(sets)
            ),
        })


stability = pd.DataFrame(
    stability_rows
)

frequency = pd.DataFrame(
    frequency_rows
)

stability.to_csv(
    OUT_DIR
    / "frozen_hybrid_feature_stability.csv",
    index=False,
)

frequency.to_csv(
    OUT_DIR
    / "frozen_hybrid_feature_frequency.csv",
    index=False,
)

display(stability)


,dataset,mean_jaccard,std_jaccard,min_jaccard,max_jaccard,pairwise_comparisons
0,Australian Credit Approval,0.962222,0.069896,0.833333,1.0,300
1,Taiwan Credit Card Default,0.943158,0.052551,0.894737,1.0,300


## 13. Final checks

Do **not** change the frozen architecture after observing these external results.

The next stage will use the saved outer probabilities for probability-calibration development and later cost-sensitive threshold optimisation.


In [15]:

assert len(results) == 2 * 25
assert len(selected_sets) == 2 * 25

expected_prediction_rows = (
    5
    * sum(
        len(df)
        for df in datasets.values()
    )
)

assert len(predictions) == expected_prediction_rows, (
    f"Expected {expected_prediction_rows} prediction rows, "
    f"found {len(predictions)}."
)

assert results["roc_auc"].between(0, 1).all()
assert results["pr_auc"].between(0, 1).all()
assert results["mcc"].between(-1, 1).all()

for column in [
    "lr_probability",
    "rf_probability",
    "xgb_probability",
    "hybrid_probability",
]:
    assert predictions[column].between(
        0, 1
    ).all()

configuration = {
    "stage": (
        "Objective 3 Step 8 - frozen hybrid ensemble replication"
    ),
    "development_dataset": "German Credit",
    "replication_datasets": list(
        datasets.keys()
    ),
    "feature_selection": (
        "Frozen Group-Aware Chi-Square Top75"
    ),
    "base_models": BASE_MODELS,
    "training_regime": "Balanced",
    "fusion": (
        "Equal arithmetic mean of positive-class probabilities"
    ),
    "threshold": 0.50,
    "external_retuning": False,
    "calibration": "not yet applied",
    "threshold_optimization": (
        "not yet applied"
    ),
}

with open(
    OUT_DIR
    / "step8_experiment_configuration.json",
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        configuration,
        f,
        indent=4,
    )

manifest = sorted(
    [
        p.name
        for p in OUT_DIR.iterdir()
        if p.is_file()
    ]
)

pd.DataFrame(
    {
        "generated_file": manifest
    }
).to_csv(
    OUT_DIR
    / "step8_output_manifest.csv",
    index=False,
)

print("=" * 80)
print("STEP 8 COMPLETED SUCCESSFULLY")
print("=" * 80)
print("Output folder:", OUT_DIR)
print("\nMost important files:")
print(" - frozen_hybrid_replication_summary.csv")
print(" - frozen_hybrid_delta_summary.csv")
print(" - frozen_hybrid_outer_predictions.csv")
print(" - frozen_hybrid_base_model_fold_results.csv")
print(" - frozen_hybrid_feature_stability.csv")


STEP 8 COMPLETED SUCCESSFULLY
Output folder: D:\PHD\Research Paper writing\3rd Obj. paper\results\frozen_hybrid_ensemble_replication

Most important files:
 - frozen_hybrid_replication_summary.csv
 - frozen_hybrid_delta_summary.csv
 - frozen_hybrid_outer_predictions.csv
 - frozen_hybrid_base_model_fold_results.csv
 - frozen_hybrid_feature_stability.csv
